In [28]:
!nvidia-smi

zsh:1: command not found: nvidia-smi


In [29]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

zsh:1: no matches found: transformers[sentencepiece]


In [30]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate

Found existing installation: transformers 5.17.0
Uninstalling transformers-5.17.0:
  Successfully uninstalled transformers-5.17.0
Found existing installation: accelerate 1.15.0
Uninstalling accelerate-1.15.0:
  Successfully uninstalled accelerate-1.15.0
  Using cached transformers-5.17.0-py3-none-any.whl.metadata (32 kB)
  Using cached accelerate-1.15.0-py3-none-any.whl.metadata (19 kB)
Using cached transformers-5.17.0-py3-none-any.whl (12.3 MB)
Using cached accelerate-1.15.0-py3-none-any.whl (394 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers] [transformers]


In [ ]:
import nltk
import pandas as pd
import torch
from datasets import load_from_disk
from tqdm import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

nltk.download("punkt")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/thepunisher/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
from transformers import PegasusForConditionalGeneration

model = PegasusForConditionalGeneration.from_pretrained("google/pegasus-xsum")
tokenizer = AutoTokenizer.from_pretrained("google/pegasus-xsum")

ARTICLE_TO_SUMMARIZE = (
    "PG&E stated it scheduled the blackouts in response to forecasts for high winds "
    "amid dry conditions. The aim is to reduce the risk of wildfires. Nearly 800 thousand customers were "
    "scheduled to be affected by the shutoffs which were expected to last through at least midday tomorrow"
)
inputs = tokenizer(ARTICLE_TO_SUMMARIZE, max_length=1024, return_tensors="pt")

# Generate Summary
summary_ids = model.generate(inputs["input_ids"])
tokenizer.batch_decode(
    summary_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]

Loading weights: 100%|██████████| 680/680 [00:00<00:00, 30608.13it/s]
[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-xsum
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


"California's largest electricity provider has turned off power to hundreds of thousands of customers."

In [ ]:
from transformers import AutoTokenizer

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
device

'mps'

## Fine Tuning

In [ ]:
model = "google/pegasus-xsum"

tokenizer = AutoTokenizer.from_pretrained(model)  # load a tokenizer

model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model).to(device)

Loading weights: 100%|██████████| 680/680 [00:00<00:00, 36878.25it/s]
[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-xsum
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [35]:
# Download & unzip data
# !wget https://github.com/GoJo-Rika/datasets/raw/refs/heads/main/summarizer-data.zip
# !unzip summarizer-data.zip

!curl -L -o summarizer-data.zip "https://github.com/GoJo-Rika/datasets/raw/refs/heads/main/summarizer-data.zip"
!unzip -o summarizer-data.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:--  0:00:01 --:--:--     0
100 7718k  100 7718k    0     0  21590      0  0:06:06  0:06:06 --:--:--  697k  0      0 --:--:--  0:00:04 --:--:--     066:54  0:00:31  3:16:23     0      0  3:29:25  0:00:33  3:28:52     0068      0  2:03:20  0:02:01  2:01:19     0  2:03:36     08:30  0:02:34  2:15:56     06  3:10:58     0   1975      0  1:06:41  0:04:33  1:02:08     00  1785      0  1:13:47  0:05:02  1:08:45     0   1756      0  1:15:00  0:05:07  1:09:53     0  0   1744      0  1:15:31  0:05:09  1:10:22     016:29  0:05:13  1:11:16     03      0  0:41:54  0:05:41  0:36:13     0   3071      0  0:42:53  0:05:50  0:37:03     0
Archive:  summarizer-data.zip
  inflating: samsum-test.csv         
  inflating: samsum-train.csv        
  inflating: samsum-validation.csv   
 extracting: samsu

In [ ]:
dataset_samsum = load_from_disk("samsum_dataset")
dataset_samsum

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14732
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
})

In [37]:
split_lengths = [len(dataset_samsum[split]) for split in dataset_samsum]

print(f"Split lengths: {split_lengths}")
print(f"Features: {dataset_samsum['train'].column_names}")
print("\nDialogue:")

print(dataset_samsum["test"][1]["dialogue"])

print("\nSummary:")
print(dataset_samsum["test"][1]["summary"])

Split lengths: [14732, 819, 818]
Features: ['id', 'dialogue', 'summary']

Dialogue:
Eric: MACHINE!
Rob: That's so gr8!
Eric: I know! And shows how Americans see Russian ;)
Rob: And it's really funny!
Eric: I know! I especially like the train part!
Rob: Hahaha! No one talks to the machine like that!
Eric: Is this his only stand-up?
Rob: Idk. I'll check.
Eric: Sure.
Rob: Turns out no! There are some of his stand-ups on youtube.
Eric: Gr8! I'll watch them now!
Rob: Me too!
Eric: MACHINE!
Rob: MACHINE!
Eric: TTYL?
Rob: Sure :)

Summary:
Eric and Rob are going to watch a stand-up on youtube.


### Preparing Data For Training For Sequence To Sequence Model

{
    'dialogue': "Hi! How are you?",
    'summary': "The speaker is asking how the other person is."
}

{
    'input_ids': [123, 456, 789, ...],  # Token IDs for the dialogue
    'attention_mask': [1, 1, 1, ...],   # Attention mask for the input
    'labels': [321, 654, 987, ...]     # Token IDs for the summary (target)
}

In [ ]:
def convert_example_to_features(example_batch):
    input_encoding = tokenizer(
        example_batch["dialogue"], max_length=512, truncation=True
    )
    target_encoding = tokenizer(
        text_target=example_batch["summary"], max_length=128, truncation=True
    )
    return {
        "input_ids": input_encoding["input_ids"],
        "attention_mask": input_encoding["attention_mask"],
        "labels": target_encoding["input_ids"],
    }


dataset_samsum_pt = dataset_samsum.map(convert_example_to_features, batched=True)

In [ ]:
dataset_samsum_pt = dataset_samsum.map(convert_example_to_features, batched=True)

In [40]:
dataset_samsum_pt["test"]

Dataset({
    features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 819
})

In [41]:
# Training

from transformers import DataCollatorForSeq2Seq

seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

In [ ]:
from transformers import Trainer, TrainingArguments

trainer_args = TrainingArguments(
    output_dir="pegasus-samsum",
    num_train_epochs=1,
    warmup_steps=500,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=1e6,
    gradient_accumulation_steps=16,
)

In [43]:
trainer = Trainer(
    model=model_pegasus,
    args=trainer_args,
    processing_class=tokenizer,
    data_collator=seq2seq_data_collator,
    train_dataset=dataset_samsum_pt["test"],
    eval_dataset=dataset_samsum_pt["validation"],
)

In [44]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss,Validation Loss
52,51.015619,2.725148


Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.33s/it]


TrainOutput(global_step=52, training_loss=52.47197547325721, metrics={'train_runtime': 392.4236, 'train_samples_per_second': 2.087, 'train_steps_per_second': 0.133, 'total_flos': 313362980069376.0, 'train_loss': 52.47197547325721, 'epoch': 1.0})

In [ ]:
# Evaluation
### lst[1, 2, 3, 4, 5, 6] -> [1, 2, 3][4, 5, 6]
def generate_batch_sized_chunks(list_of_elements, batch_size):
    """Split the dataset into smaller batches that we can process simultaneously
    Yield successive batch-sized chunks from list_of_elements."""
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i : i + batch_size]


def calculate_metric_on_test_ds(
    dataset,
    metric,
    model,
    tokenizer,
    batch_size=16,
    device=device,
    column_text="dialogue",
    column_summary="summary",
):
    model_device = next(model.parameters()).device
    article_batches = list(
        generate_batch_sized_chunks(dataset[column_text], batch_size)
    )
    target_batches = list(
        generate_batch_sized_chunks(dataset[column_summary], batch_size)
    )

    for article_batch, target_batch in tqdm(
        zip(article_batches, target_batches), total=len(article_batches)
    ):
        inputs = tokenizer(
            article_batch,
            max_length=512,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        summaries = model.generate(
            input_ids=inputs["input_ids"].to(model_device),
            attention_mask=inputs["attention_mask"].to(device),
            length_penalty=0.8,
            num_beams=8,
            max_length=128,
        )
        """ Parameter for length penalty ensures that the model does not generate sequences that are too long. """

        # Finally, we decode the generated texts,
        # replace the token, and add the decoded texts with the references to the metric.
        decoded_summaries = [
            tokenizer.decode(
                s, skip_special_tokens=True, clean_up_tokenization_spaces=True
            )
            for s in summaries
        ]

        #   decoded_summaries = [d.replace("", " ") for d in decoded_summaries]

        metric.add_batch(predictions=decoded_summaries, references=target_batch)

    # Finally compute and return the ROUGE scores.
    score = metric.compute()
    return score


In [46]:
!pip install evaluate

In [51]:
import evaluate

rouge_metric = evaluate.load("rouge")
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
# rouge_metric = load_metric("rouge")

In [52]:
rouge_metric

EvaluationModule(name: "rouge", module_type: "metric", features: [{'predictions': Value('string'), 'references': List(Value('string'))}, {'predictions': Value('string'), 'references': Value('string')}], usage: """
Calculates average rouge scores for a list of hypotheses and references
Args:
    predictions: list of predictions to score. Each prediction
        should be a string with tokens separated by spaces.
    references: list of reference for each prediction. Each
        reference should be a string with tokens separated by spaces.
    rouge_types: A list of rouge types to calculate.
        Valid names:
        `"rouge{n}"` (e.g. `"rouge1"`, `"rouge2"`) where: {n} is the n-gram based scoring,
        `"rougeL"`: Longest common subsequence based scoring.
        `"rougeLsum"`: rougeLsum splits text using `"
"`.
        See details in https://github.com/huggingface/datasets/issues/617
    use_stemmer: Bool indicating whether Porter stemmer should be used to strip word suffixes.
 

In [ ]:
score = calculate_metric_on_test_ds(
    dataset_samsum["test"][0:10],
    rouge_metric,
    trainer.model,
    tokenizer,
    batch_size=2,
    column_text="dialogue",
    column_summary="summary",
)

# Directly use the scores without accessing fmeasure or mid
rouge_dict = {rn: score[rn] for rn in rouge_names}

# Convert the dictionary to a DataFrame for easy visualization
pd.DataFrame(rouge_dict, index=["pegasus"])


100%|██████████| 5/5 [00:34<00:00,  6.99s/it]


,rouge1,rouge2,rougeL,rougeLsum
pegasus,0.211768,0.030655,0.155084,0.155934


In [61]:
## Save model
model_pegasus.save_pretrained("pegasus-samsum-model")

Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.91s/it]


In [62]:
## Save tokenizer
tokenizer.save_pretrained("tokenizer")


('tokenizer/tokenizer_config.json', 'tokenizer/tokenizer.json')

In [64]:
# Load
tokenizer = AutoTokenizer.from_pretrained("tokenizer")

In [ ]:
sample_text = dataset_samsum["test"][0]["dialogue"]
reference = dataset_samsum["test"][0]["summary"]

# Option A: model fine-tuned in memory
model_to_use = trainer.model
tok = tokenizer

# Option B: if you saved it on disk
# model_to_use = AutoModelForSeq2SeqLM.from_pretrained("pegasus-samsum")
# tok = AutoTokenizer.from_pretrained("pegasus-samsum")

model_device = next(model_to_use.parameters()).device
inputs = tok(sample_text, max_length=512, truncation=True, return_tensors="pt").to(
    model_device
)

summary_ids = model_to_use.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_length=128,
    num_beams=8,
    length_penalty=0.8,
)
gen_summary = tok.decode(summary_ids[0], skip_special_tokens=True)

print("Dialogue:")
print(sample_text)
print("\nReference Summary:")
print(reference)
print("\nModel Summary:")
print(gen_summary)

Dialogue:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

Reference Summary:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.

Model Summary:
Amanda: Hey Hannah, do you have Betty's number? Amanda: Ask Larry Amanda: He called her last time we were at the park together Hannah: I don't know him well Hannah: file_gif> Amanda: Don't be shy, he's very nice Hannah: If you say so..
